# 00 - Dataset audit, duplicate/leakage detection, split definitions

**What this notebook does**
1. Resolves the dataset root (local `Data/` or `/kaggle/input/<slug>/`).
2. Indexes every image and prints class counts per split.
3. Hashes every file to find duplicates and train/test leakage (LESSON 4).
4. Writes the two split definitions every later notebook depends on:
   * `outputs/splits/faithful_split.csv` - the paper-comparable split, duplicates left in
   * `outputs/splits/clean_split.csv` - deduplicated, leakage-free, group-aware stratified
5. Saves a written audit report to `outputs/reports/dataset_audit.md`.

**What must already exist**: the Kaggle *Chest CT-Scan images* dataset - attached as a Kaggle input
(Add Input -> search "Chest CT-Scan images" by Mohamed Hany), or present in a local `Data/` folder
with `train/`, `valid/`, `test/` subfolders.

**Run this notebook first.** Notebooks 01-07 all call `load_split()` and will raise a clear
`FileNotFoundError` if these CSVs do not exist yet.

**What "looks right"**: ~1000 images total across 4 classes (roughly 613 train / 72 val / 315 test).
If you see far fewer, dataset path resolution failed - check the printed data root first.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))  # repo root, so `import src` works

from src.config import *
from src.data_utils import resolve_data_root

print("repo root on sys.path:", os.path.abspath(".."))
for k, v in describe_environment().items():
    print(f"  {k}: {v}")
print()
print("created output dirs:")
for k, v in ensure_dirs().items():
    print(f"  {k}: {v}")

repo root on sys.path: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-
  PROJECT_ROOT: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-
  IS_KAGGLE: False
  OUTPUT_ROOT: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/outputs
  MODELS_DIR: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/models
  SEED: 42
  IMG_SIZE: (224, 224)
  BATCH_SIZE: 32
  CLASS_NAMES: ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']

created output dirs:
  OUTPUT_ROOT: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/outputs
  FIGURES_DIR: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/outputs/figures
  HISTORY_DIR: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/outputs/history
  REPORTS_DIR: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Pr

## 1. Resolve the dataset root

**Looks right**: a path that contains `train/`, `valid/`, `test/`. On Kaggle it will start with
`/kaggle/input/`. If this cell raises, the dataset is not attached or the folder name did not match
the `ctscan` / `chest` pattern - read the error message, it lists what it actually found.

In [2]:
DATA_ROOT = resolve_data_root()
print("DATA_ROOT:", DATA_ROOT)
print("contents:", sorted(os.listdir(DATA_ROOT)))

DATA_ROOT: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/Data
contents: ['test', 'train', 'valid']


## 2. Index every image

Builds one row per file. Note the folder-name normalisation: `train/` and `valid/` use long staging
names (`adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib`) while `test/` uses short ones
(`adenocarcinoma`); both map onto the same canonical class.

**Looks right**: ~1000 rows, exactly 4 unique classes, 3 unique splits.

In [3]:
import pandas as pd
from src.data_utils import index_dataset, class_counts

df = index_dataset(DATA_ROOT)

print("total images:", len(df))
print("classes     :", sorted(df['class'].unique()))
print("splits      :", sorted(df['orig_split'].unique()))
df.head(3)

total images: 1000
classes     : ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']
splits      : ['test', 'train', 'val']


,filepath,filename,class,label,orig_split,folder
0,/mnt/c/Users/shrey/OneDrive/sem7/Medical Image...,000108 (3).png,adenocarcinoma,0,test,adenocarcinoma
1,/mnt/c/Users/shrey/OneDrive/sem7/Medical Image...,000109 (2).png,adenocarcinoma,0,test,adenocarcinoma
2,/mnt/c/Users/shrey/OneDrive/sem7/Medical Image...,000109 (4).png,adenocarcinoma,0,test,adenocarcinoma


In [4]:
counts = class_counts(df)
print("images per class per original split")
print(counts)
print()
print("per-split totals:")
print(df['orig_split'].value_counts().reindex(['train', 'val', 'test']))

images per class per original split
orig_split               test  train  val  total
class                                           
adenocarcinoma            120    195   23    338
large.cell.carcinoma       51    115   21    187
normal                     54    148   13    215
squamous.cell.carcinoma    90    155   15    260

per-split totals:
orig_split
train    613
val       72
test     315
Name: count, dtype: int64


In [5]:
# The raw folder names behind each canonical class - confirms normalisation worked.
print(df.groupby(['class', 'orig_split'])['folder'].unique().to_string())

class                    orig_split
adenocarcinoma           test                                           [adenocarcinoma]
                         train              [adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib]
                         val                [adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib]
large.cell.carcinoma     test                                     [large.cell.carcinoma]
                         train           [large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa]
                         val             [large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa]
normal                   test                                                   [normal]
                         train                                                  [normal]
                         val                                                    [normal]
squamous.cell.carcinoma  test                                  [squamous.cell.carcinoma]
                         train         [squamous.cell.carcinoma_left.hilum

## 3. Image properties

Reads a small sample of files to confirm they decode and to see the native size/mode spread.
**Looks right**: PNG/JPEG images, mixed native sizes (they get resized to 224x224 later), mode `RGB`
or `L`. No decode errors.

In [6]:
from PIL import Image

sample = df.groupby('class', group_keys=False).head(25)
rows = []
for p in sample['filepath']:
    with Image.open(p) as im:
        rows.append({'width': im.size[0], 'height': im.size[1], 'mode': im.mode,
                     'format': im.format})
props = pd.DataFrame(rows)
print('sampled', len(props), 'files')
print(props[['width', 'height']].describe().loc[['min', 'max', 'mean']])
print()
print('modes  :', props['mode'].value_counts().to_dict())
print('formats:', props['format'].value_counts().to_dict())

sampled 100 files
       width  height
min   360.00  245.00
max   629.00  491.00
mean  442.87  329.94

modes  : {'RGBA': 100}
formats: {'PNG': 100}


## 4. Hash every file (duplicate detection)

MD5 over raw file bytes. This reads ~1000 small files and takes well under a minute; it trains
nothing.

**Looks right**: `unique hashes` noticeably smaller than `total images` - this dataset is known to
contain duplicates (filenames like `10 (2) - Copy.png` are the giveaway).

In [7]:
from src.data_utils import add_hashes

df = add_hashes(df)
print('total images :', len(df))
print('unique hashes:', df['hash'].nunique())
print('redundant    :', len(df) - df['hash'].nunique())

  hashed 200/1000 files
  hashed 400/1000 files
  hashed 600/1000 files
  hashed 800/1000 files
  hashed 1000/1000 files
total images : 1000
unique hashes: 847
redundant    : 153


## 5. Leakage audit (LESSON 4)

A *duplicate* is any file whose content hash appears more than once. A *leaked* file is a duplicate
whose hash appears in more than one split - the model can memorise it during training and be
rewarded for that memorisation at test time.

**Looks right**: a non-zero number of duplicates, concentrated in the `normal` class (the earlier
attempt found 153 duplicate files). If this reports zero duplicates, hashing silently failed.

In [8]:
from src.data_utils import audit_leakage

audit = audit_leakage(df)
for k, v in audit.items():
    if k.endswith('_rows'):
        continue
    print(f'{k}: {v}')

n_files: 1000
n_unique_hashes: 847
n_duplicate_groups: 59
n_files_in_duplicate_groups: 212
n_redundant_files: 153
n_leaked_files: 120
n_train_test_leaked_files: 101
duplicates_per_class: {'adenocarcinoma': 2, 'large.cell.carcinoma': 1, 'normal': 204, 'squamous.cell.carcinoma': 5}
leaked_per_class: {'adenocarcinoma': 0, 'large.cell.carcinoma': 0, 'normal': 120, 'squamous.cell.carcinoma': 0}


In [9]:
# Where do the duplicates sit? (split x class)
dups = audit['duplicate_rows']
if len(dups):
    print(pd.crosstab(dups['class'], dups['orig_split']))
    print()
    print('largest duplicate groups:')
    print(dups['hash'].value_counts().head(5))
else:
    print('No duplicates found - unexpected for this dataset, re-check hashing.')

orig_split               test  train  val
class                                    
adenocarcinoma              0      2    0
large.cell.carcinoma        0      1    0
normal                     46    148   10
squamous.cell.carcinoma     0      5    0

largest duplicate groups:
hash
be891546a445986a881ba74696f8a492    10
2a22fd805d0db68736491698305c25d8     9
489afa049dc03363c587ef4f1c5f1d70     9
81a47ae233d917ad8fa9d66c9f4ba1bd     9
c5b028c8ce7837ec365dfd4606eea527     9
Name: count, dtype: int64


In [10]:
# Files that appear in BOTH train and test - the actual leakage.
leaked = audit['leaked_rows']
if len(leaked):
    print(pd.crosstab(leaked['class'], leaked['orig_split']))
    print()
    print('example leaked group:')
    h = leaked['hash'].iloc[0]
    print(leaked[leaked['hash'] == h][['orig_split', 'class', 'filename']].to_string(index=False))
else:
    print('No cross-split leakage detected.')

orig_split  test  train  val
class                       
normal        46     64   10

example leaked group:
orig_split  class          filename
      test normal 10 (2) - Copy.png
      test normal        10 (2).png
     train normal 10 (2) - Copy.png
     train normal        10 (2).png


## 6. Build and save the `faithful` split

The dataset's own `train/valid/test` folders, untouched - duplicates included as-is. This is the
split whose numbers are comparable with the paper.

**Looks right**: the same counts as section 2.

In [11]:
from src.data_utils import build_faithful_split, save_split, split_counts, imbalance_ratio

faithful = build_faithful_split(df)
path_faithful = save_split(faithful, 'faithful')

print('saved:', path_faithful)
print(split_counts(faithful))
print()
print('train imbalance ratio (max/min):', round(imbalance_ratio(faithful, 'train'), 2))

saved: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/outputs/splits/faithful_split.csv
split                    train  val  test  total
class                                           
adenocarcinoma             195   23   120    338
large.cell.carcinoma       115   21    51    187
normal                     148   13    54    215
squamous.cell.carcinoma    155   15    90    260

train imbalance ratio (max/min): 1.7


## 7. Build and save the `clean` split

One representative per content hash, then a stratified 70/10/20 re-split. Because the grouping key
is the hash and only one member of each group survives, no duplicate can straddle two splits -
leakage-freedom holds by construction and is asserted before saving.

**This is NOT a replication of the paper's protocol.** Label its results as a robustness /
generalisation experiment everywhere they appear.

**Looks right**: fewer total images than `faithful`, and a train imbalance ratio above ~2 (LESSON 5
- `normal` shrinks the most), which is why class weighting is on by default for this variant.

In [12]:
from src.data_utils import build_clean_split, assert_no_leakage

clean = build_clean_split(df, fractions=CLEAN_SPLIT_FRACTIONS, seed=SEED)
path_clean = save_split(clean, 'clean')

print('saved:', path_clean)
print('images: faithful =', len(faithful), '| clean =', len(clean),
      '| removed =', len(faithful) - len(clean))
print()
print(split_counts(clean))

saved: /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/outputs/splits/clean_split.csv
images: faithful = 1000 | clean = 847 | removed = 153

split                    train  val  test  total
class                                           
adenocarcinoma             235   34    68    337
large.cell.carcinoma       131   19    37    187
normal                      46    7    13     66
squamous.cell.carcinoma    180   25    52    257


In [13]:
print('clean train imbalance ratio (max/min):', round(imbalance_ratio(clean, 'train'), 2))
print()
print('clean train class counts:')
print(clean[clean['split'] == 'train']['class'].value_counts().reindex(CLASS_NAMES))
print()
from src.train_utils import compute_class_weights
print('class_weight that will be used for clean training:')
print(compute_class_weights(clean[clean['split'] == 'train']['label'].values))

clean train imbalance ratio (max/min): 5.11

clean train class counts:
class
adenocarcinoma             235
large.cell.carcinoma       131
normal                      46
squamous.cell.carcinoma    180
Name: count, dtype: int64

class_weight that will be used for clean training:
{0: 0.6297872340425532, 1: 1.1297709923664123, 2: 3.217391304347826, 3: 0.8222222222222222}


## 8. Verify the split guarantees

**Looks right**: `clean` passes the no-leakage assertion; `faithful` is *expected* to fail it - that
failure is the documented property of the paper-comparable split, not a bug.

In [14]:
try:
    assert_no_leakage(clean)
    print('clean   : PASS - no content hash spans two splits')
except AssertionError as e:
    print('clean   : FAIL -', e)

try:
    assert_no_leakage(faithful)
    print('faithful: no leakage found (unexpected for this dataset)')
except AssertionError as e:
    print('faithful: leakage present as expected -', e)

clean   : PASS - no content hash spans two splits
faithful: leakage present as expected - 22 content hashes appear in more than one split - the split is NOT leakage-free.


## 9. Write the audit report

Saves everything above to `outputs/reports/dataset_audit.md` so the numbers can be quoted in the
README without re-running the notebook.

In [15]:
report_path = REPORTS_DIR / 'dataset_audit.md'

lines = [
    '# Dataset audit',
    '',
    f'- data root: `{DATA_ROOT}`',
    f'- total images: {len(df)}',
    f'- unique content hashes: {df["hash"].nunique()}',
    f'- duplicate groups: {audit["n_duplicate_groups"]}',
    f'- files in duplicate groups: {audit["n_files_in_duplicate_groups"]}',
    f'- redundant (removable) files: {audit["n_redundant_files"]}',
    f'- files leaking across splits: {audit["n_leaked_files"]}',
    f'- files leaking specifically train<->test: {audit["n_train_test_leaked_files"]}',
    '',
    '## Duplicates per class',
    '',
    *(f'- {k}: {v}' for k, v in audit['duplicates_per_class'].items()),
    '',
    '## Class counts per original split',
    '',
    '```',
    counts.to_string(),
    '```',
    '',
    '## faithful split (paper-comparable, duplicates kept)',
    '',
    '```',
    split_counts(faithful).to_string(),
    '```',
    '',
    '## clean split (deduplicated, leakage-free - robustness experiment, NOT a replication)',
    '',
    '```',
    split_counts(clean).to_string(),
    '```',
    '',
    f'- clean train imbalance ratio (max/min): {imbalance_ratio(clean, "train"):.2f}',
    '- class weighting is enabled by default for `clean` training runs.',
]

report_path.write_text('\n'.join(lines))
print('wrote', report_path)
print()
print('\n'.join(lines[:14]))

wrote /mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/outputs/reports/dataset_audit.md

# Dataset audit

- data root: `/mnt/c/Users/shrey/OneDrive/sem7/Medical Image/Project/Lung-Cancer-Classification-/Data`
- total images: 1000
- unique content hashes: 847
- duplicate groups: 59
- files in duplicate groups: 212
- redundant (removable) files: 153
- files leaking across splits: 120
- files leaking specifically train<->test: 101

## Duplicates per class

- adenocarcinoma: 2


## 10. Initialise the results files

Creates `results_table.csv`, `experiments_log.csv` and `ablation_dropout.csv` with headers only if
they do not already exist. Never overwrites real results unless you pass `overwrite=True`.

In [16]:
from src.evaluate_utils import init_results_files, load_results

print('created (empty, header only):', init_results_files())
for kind in ('canonical', 'experiment', 'ablation'):
    print(f'{kind:11s} rows so far: {len(load_results(kind))}')

print('\nDone. Next: 01_preprocessing_check.ipynb')

created (empty, header only): {}
canonical   rows so far: 0
experiment  rows so far: 4
ablation    rows so far: 2

Done. Next: 01_preprocessing_check.ipynb
